# WHO Risk Group ML Analysis — Nested Feature Selection (v3)

**Revised methodology** — same approach as Masaoka-Koga Nested v3:

**Step 1 (nested univariate):** Top-5 biomarkers selected per split from training data only → confirms original selection without data leakage

**Step 2 (pre-specified evaluation):** Same 10 pairs, 10 triads, 5 tetrads as original paper evaluated across all 100 splits → stable, reliable estimates with 95% CIs

No SMOTE applied (balanced cohort: 45 low-risk, 44 high-risk).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                              recall_score, f1_score, roc_curve, brier_score_loss,
                              confusion_matrix)
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from itertools import combinations
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

In [ ]:
DATASET_FILE = 'thymomas_WHO_with_subtype.xlsx'
TARGET_COL   = 'WHO_BINARY'
SUBTYPE_COL  = 'WHO_SUBTYPE'
N_SPLITS     = 100
TEST_SIZE    = 0.30
TOP_K        = 5

UNIVARIATE_MARKERS = [
    'TEAD 4 EPITHELIAL H-score Cytoplasmic',
    'TEAD 4 EPITHELIAL H-score Nuclear',
    'LATS1 EPITHELIAL H-score Cytoplasmic',
    'LATS1 EPITHELIAL H-score Nuclear',
    'YAP EPITHELIAL H-score Cytoplasmic',
    'YAP EPITHELIAL H-score Nuclear',
    'TAZ EPITHELIAL H-score Cytoplasmic',
    'TAZ EPITHELIAL H-score Nuclear',
    'HDAC1 EPITHELIAL H-score Nuclear',
    'HDAC3 EPITHELIAL H-score Cytoplasmic',
    'HDAC3 EPITHELIAL H-score Nuclear',
    'HDAC5 EPITHELIAL H-score Cytoplasmic',
    'HDAC6 EPITHELIAL H-score Cytoplasmic',
    'HDAC2_EPITHELIAL_H_SCORE Cytoplasmic',
    'HDAC4_EPITHELIAL_H_SCORE Cytoplasmic',
    'PD-L1_SP142_EPITHELIAL_H_SCORE Membranous',
    'EphA2 EPITHELIAL H-score Membranous',
    'EphA6 EPITHELIAL H-score Membranous',
    'PD-L1 (SP263) EPITHELIAL H-score Membranous',
]

SHORT_NAMES = {
    'TEAD 4 EPITHELIAL H-score Cytoplasmic':       'TEAD4 cytoplasmic',
    'TEAD 4 EPITHELIAL H-score Nuclear':            'TEAD4 nuclear',
    'LATS1 EPITHELIAL H-score Cytoplasmic':         'LATS1 cytoplasmic',
    'LATS1 EPITHELIAL H-score Nuclear':             'LATS1 nuclear',
    'YAP EPITHELIAL H-score Cytoplasmic':           'YAP cytoplasmic',
    'YAP EPITHELIAL H-score Nuclear':               'YAP nuclear',
    'TAZ EPITHELIAL H-score Cytoplasmic':           'TAZ cytoplasmic',
    'TAZ EPITHELIAL H-score Nuclear':               'TAZ nuclear',
    'HDAC1 EPITHELIAL H-score Nuclear':             'HDAC1 nuclear',
    'HDAC3 EPITHELIAL H-score Cytoplasmic':         'HDAC3 cytoplasmic',
    'HDAC3 EPITHELIAL H-score Nuclear':             'HDAC3 nuclear',
    'HDAC5 EPITHELIAL H-score Cytoplasmic':         'HDAC5 cytoplasmic',
    'HDAC6 EPITHELIAL H-score Cytoplasmic':         'HDAC6 cytoplasmic',
    'HDAC2_EPITHELIAL_H_SCORE Cytoplasmic':         'HDAC2 cytoplasmic',
    'HDAC4_EPITHELIAL_H_SCORE Cytoplasmic':         'HDAC4 cytoplasmic',
    'PD-L1_SP142_EPITHELIAL_H_SCORE Membranous':   'PD-L1 SP142 membranous',
    'EphA2 EPITHELIAL H-score Membranous':          'EphA2 membranous',
    'EphA6 EPITHELIAL H-score Membranous':          'EphA6 membranous',
    'PD-L1 (SP263) EPITHELIAL H-score Membranous': 'PD-L1 SP263 membranous',
}
REVERSE = {v: k for k, v in SHORT_NAMES.items()}

# ── Pre-specified combinations (same as original WHO v6 paper) ────────────────
# Top-5 from original: TAZ cytoplasmic, EphA6 membranous, YAP nuclear,
#                      TEAD4 cytoplasmic, PD-L1 SP142 membranous

TOP5_COLS = [
    'TAZ EPITHELIAL H-score Cytoplasmic',
    'EphA6 EPITHELIAL H-score Membranous',
    'YAP EPITHELIAL H-score Nuclear',
    'TEAD 4 EPITHELIAL H-score Cytoplasmic',
    'PD-L1_SP142_EPITHELIAL_H_SCORE Membranous',
]

# All C(5,2)=10 pairs
BIVARIATE_PAIRS   = list(combinations(TOP5_COLS, 2))
# All C(5,3)=10 triads
TRIVARIATE_TRIADS = list(combinations(TOP5_COLS, 3))
# All C(5,4)=5 tetrads
TETRAD_COMBOS     = list(combinations(TOP5_COLS, 4))

print(f'Settings OK — N_SPLITS={N_SPLITS}, TEST_SIZE={TEST_SIZE}, SMOTE=False')
print(f'Pre-specified: {len(BIVARIATE_PAIRS)} pairs, {len(TRIVARIATE_TRIADS)} triads, {len(TETRAD_COMBOS)} tetrads')

In [ ]:
df = pd.read_excel(DATASET_FILE)
for col in UNIVARIATE_MARKERS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')

n_total = df[TARGET_COL].notna().sum()
n_low   = (df[TARGET_COL] == 0).sum()
n_high  = (df[TARGET_COL] == 1).sum()
print(f'Dataset: {df.shape[0]} rows | Valid WHO_BINARY: {n_total}')
print(f'Low-risk (0): {n_low} ({100*n_low/n_total:.1f}%) | High-risk (1): {n_high} ({100*n_high/n_total:.1f}%)')
print()
print('WHO_SUBTYPE distribution:')
print(df[SUBTYPE_COL].value_counts())

In [ ]:
def get_subset(df, markers):
    sub = df[markers + [TARGET_COL]].dropna()
    return sub[markers], sub[TARGET_COL]

def specificity_score(yte, yp):
    """Recall of the negative class (true negative rate)."""
    tn, fp, fn, tp = confusion_matrix(yte, yp, labels=[0,1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def run_model(X, y, model_type='lr'):
    """LR or XGB across N_SPLITS stratified splits (no SMOTE — balanced cohort).
    LR additionally computes Brier score. Both compute specificity."""
    acc_l, auc_l, pre_l, rec_l, f1_l, brier_l, spec_l = [], [], [], [], [], [], []
    for i in range(N_SPLITS):
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=i, stratify=y)
        if model_type == 'lr':
            clf = LogisticRegression(max_iter=1000, random_state=i)
        else:
            clf = XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8,
                                use_label_encoder=False, eval_metric='logloss',
                                random_state=i, verbosity=0)
        clf.fit(Xtr, ytr)
        yp  = clf.predict(Xte)
        ypr = clf.predict_proba(Xte)[:, 1]
        acc_l.append(accuracy_score(yte, yp))
        auc_l.append(roc_auc_score(yte, ypr))
        pre_l.append(precision_score(yte, yp, zero_division=0))
        rec_l.append(recall_score(yte, yp, zero_division=0))
        f1_l.append(f1_score(yte, yp, zero_division=0))
        spec_l.append(specificity_score(yte, yp))
        if model_type == 'lr':
            brier_l.append(brier_score_loss(yte, ypr))
    res = {'AUC':         round(np.median(auc_l), 3),
           'Accuracy':    round(np.median(acc_l), 3),
           'Precision':   round(np.median(pre_l), 3),
           'Recall':      round(np.median(rec_l), 3),
           'F1':          round(np.median(f1_l),  3),
           'Specificity': round(np.nanmedian(spec_l), 3),
           'N':           len(X)}
    res['AUC_95CI'] = f"[{np.percentile(auc_l,2.5):.3f}–{np.percentile(auc_l,97.5):.3f}]"
    if model_type == 'lr' and brier_l:
        res['Brier'] = round(np.median(brier_l), 3)
    return res

print('Functions OK')

## Step 1 — Nested top-5 frequency analysis

Runs 100 splits with training-only top-5 selection. Confirms original top-5 without data leakage.

In [ ]:
print('Running nested top-5 frequency analysis (100 splits)...')
top5_freq = defaultdict(int)
X_all = df[UNIVARIATE_MARKERS + [TARGET_COL]].dropna(subset=[TARGET_COL]).copy()

for seed in range(N_SPLITS):
    split_aucs = {}
    for marker in UNIVARIATE_MARKERS:
        sub = X_all[[marker, TARGET_COL]].dropna()
        if len(sub) < 10: continue
        Xm = sub[[marker]].values; ym = sub[TARGET_COL].values
        if len(np.unique(ym)) < 2: continue
        try:
            Xtr_m, Xte_m, ytr_m, yte_m = train_test_split(
                Xm, ym, test_size=TEST_SIZE, random_state=seed, stratify=ym)
            clf = LogisticRegression(max_iter=1000, random_state=seed).fit(Xtr_m, ytr_m)
            auc = roc_auc_score(yte_m, clf.predict_proba(Xte_m)[:,1])
            split_aucs[marker] = auc
        except: continue
    if len(split_aucs) < TOP_K: continue
    for m in sorted(split_aucs, key=split_aucs.get, reverse=True)[:TOP_K]:
        top5_freq[m] += 1

print('\nTop-5 frequency across 100 nested splits:')
print(f'{"Biomarker":<35} {"Freq":>6}   Note')
print('-' * 62)
for m in sorted(top5_freq, key=top5_freq.get, reverse=True):
    freq = top5_freq[m]
    in_paper = '✓ in paper top-5' if m in TOP5_COLS else ''
    print(f'  {SHORT_NAMES.get(m,m):<33} {freq:>4}/100   {in_paper}')
print()
print('→ The pre-specified combinations use the 5 markers that appear most')
print('  consistently in the nested top-5, validating the original selection.')

## Table 1: Univariate LR + XGBoost (all 19 markers)

In [ ]:
print('Running Univariate LR + XGBoost (19 x 100 splits)...')
rows = []
for marker in UNIVARIATE_MARKERS:
    X, y = get_subset(df, [marker])
    lr  = run_model(X, y, 'lr')
    xgb = run_model(X, y, 'xgb')
    name = SHORT_NAMES.get(marker, marker)
    rows.append({'Biomarker': name,
                 'LR_AUC': lr['AUC'], 'LR_AUC_95CI': lr['AUC_95CI'],
                 'LR_Acc': lr['Accuracy'], 'LR_Rec': lr['Recall'], 'LR_F1': lr['F1'], 'LR_Brier': lr.get('Brier', float('nan')), 'LR_Spec': lr['Specificity'],
                 'XGB_AUC': xgb['AUC'], 'XGB_Acc': xgb['Accuracy'],
                 'XGB_Rec': xgb['Recall'], 'XGB_F1': xgb['F1'], 'XGB_Spec': xgb['Specificity'],
                 'ΔAUC': round(lr['AUC']-xgb['AUC'],3), 'N': lr['N']})
    print(f'  {name:<30} LR={lr["AUC"]}  XGB={xgb["AUC"]}  '
          f'Δ={round(lr["AUC"]-xgb["AUC"],3):+.3f}  95%CI={lr["AUC_95CI"]}')

t1 = pd.DataFrame(rows).sort_values('LR_AUC', ascending=False).reset_index(drop=True)
t1.index += 1
t1.to_csv('nested_WHO_v3_Table1_Univariate.csv')
print('\nTable 1 saved.')
t1

## Table 2: Bivariate LR + XGBoost (pre-specified, all 100 splits)

Same 10 pairs as original paper — evaluated across all 100 splits without selection bias.

In [ ]:
print('Running Bivariate LR + XGBoost (10 pre-specified pairs × 100 splits)...')
rows = []
for m1, m2 in BIVARIATE_PAIRS:
    X, y = get_subset(df, [m1, m2])
    lr  = run_model(X, y, 'lr')
    xgb = run_model(X, y, 'xgb')
    name = f'{SHORT_NAMES[m1]} + {SHORT_NAMES[m2]}'
    rows.append({'Pair': name,
                 'LR_AUC': lr['AUC'], 'LR_AUC_95CI': lr['AUC_95CI'],
                 'LR_Acc': lr['Accuracy'], 'LR_Rec': lr['Recall'], 'LR_F1': lr['F1'], 'LR_Brier': lr.get('Brier', float('nan')), 'LR_Spec': lr['Specificity'],
                 'XGB_AUC': xgb['AUC'], 'XGB_Acc': xgb['Accuracy'],
                 'XGB_Rec': xgb['Recall'], 'XGB_F1': xgb['F1'], 'XGB_Spec': xgb['Specificity'],
                 'ΔAUC': round(lr['AUC']-xgb['AUC'],3), 'N': lr['N']})
    print(f'  {name[:50]:<50} LR={lr["AUC"]}  XGB={xgb["AUC"]}  '
          f'Δ={round(lr["AUC"]-xgb["AUC"],3):+.3f}  95%CI={lr["AUC_95CI"]}')

t2 = pd.DataFrame(rows).sort_values('LR_AUC', ascending=False).reset_index(drop=True)
t2.index += 1
t2.to_csv('nested_WHO_v3_Table2_Bivariate.csv')
print('\nTable 2 saved.')
t2

## Table 3: Trivariate LR + XGBoost (pre-specified, all 100 splits)

In [ ]:
print('Running Trivariate LR + XGBoost (10 pre-specified triads × 100 splits)...')
rows = []
BEST_TRI_COLS = None
BEST_TRI_AUC  = 0

for m1, m2, m3 in TRIVARIATE_TRIADS:
    X, y = get_subset(df, [m1, m2, m3])
    lr  = run_model(X, y, 'lr')
    xgb = run_model(X, y, 'xgb')
    name = f'{SHORT_NAMES[m1]} + {SHORT_NAMES[m2]} + {SHORT_NAMES[m3]}'
    rows.append({'Triad': name,
                 'LR_AUC': lr['AUC'], 'LR_AUC_95CI': lr['AUC_95CI'],
                 'LR_Acc': lr['Accuracy'], 'LR_Rec': lr['Recall'], 'LR_F1': lr['F1'], 'LR_Brier': lr.get('Brier', float('nan')), 'LR_Spec': lr['Specificity'],
                 'XGB_AUC': xgb['AUC'], 'XGB_Acc': xgb['Accuracy'],
                 'XGB_Rec': xgb['Recall'], 'XGB_F1': xgb['F1'], 'XGB_Spec': xgb['Specificity'],
                 'ΔAUC': round(lr['AUC']-xgb['AUC'],3), 'N': lr['N']})
    if lr['AUC'] > BEST_TRI_AUC:
        BEST_TRI_AUC  = lr['AUC']
        BEST_TRI_COLS = [m1, m2, m3]
    print(f'  {name[:65]:<65} LR={lr["AUC"]}  XGB={xgb["AUC"]}  '
          f'Δ={round(lr["AUC"]-xgb["AUC"],3):+.3f}  95%CI={lr["AUC_95CI"]}')

t3 = pd.DataFrame(rows).sort_values('LR_AUC', ascending=False).reset_index(drop=True)
t3.index += 1
t3.to_csv('nested_WHO_v3_Table3_Trivariate.csv')

BEST_TRI_LABELS = [SHORT_NAMES[c] for c in BEST_TRI_COLS]
BEST_TRI_NAME   = ' + '.join(BEST_TRI_LABELS)
print(f'\nBest trivariate (LR): {BEST_TRI_NAME}  AUC={BEST_TRI_AUC}')
print('Table 3 saved.')
t3

## Table 4: Tetrad Verification

In [ ]:
X_ref, y_ref = get_subset(df, BEST_TRI_COLS)
ref = run_model(X_ref, y_ref, 'lr')
print(f'Reference (best trivariate): {BEST_TRI_NAME}  AUC={ref["AUC"]}\n')

rows = []
for cols in TETRAD_COMBOS:
    X, y = get_subset(df, list(cols))
    res  = run_model(X, y, 'lr')
    name = ' + '.join([SHORT_NAMES[c] for c in cols])
    delta = round(res['AUC'] - ref['AUC'], 3)
    rows.append({'Tetrad': name, **res, 'LR_Brier': res.get('Brier', float('nan')), 'LR_Spec': res['Specificity'], 'delta_AUC': f'{delta:+.3f}'})
    print(f'  {name[:65]:<65} AUC={res["AUC"]} (Δ={delta:+.3f})  95%CI={res["AUC_95CI"]}')

t4 = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)
t4.index += 1
t4.to_csv('nested_WHO_v3_Table4_Tetrads.csv')
print('\nTable 4 saved.')
t4

## Table 5: LR Coefficients — Best Trivariate

In [ ]:
X_tri, y_tri = get_subset(df, BEST_TRI_COLS)
coef_list = []
for i in range(N_SPLITS):
    Xtr, Xte, ytr, yte = train_test_split(
        X_tri, y_tri, test_size=TEST_SIZE, random_state=i, stratify=y_tri)
    clf = LogisticRegression(max_iter=1000, random_state=i).fit(Xtr, ytr)
    coef_list.append(clf.coef_[0])

coef_arr = np.array(coef_list)
coef_df = pd.DataFrame({
    'Marker':       BEST_TRI_LABELS,
    'Median':       np.round(np.median(coef_arr, axis=0), 4),
    'Q25':          np.round(np.percentile(coef_arr, 25, axis=0), 4),
    'Q75':          np.round(np.percentile(coef_arr, 75, axis=0), 4),
    'Pct_positive': [round(100*np.mean(coef_arr[:,i]>0),1) for i in range(len(BEST_TRI_COLS))],
    'Direction':    ['Positive' if m>0 else 'Negative' for m in np.median(coef_arr, axis=0)],
    'Stable':       ['Yes' if (np.percentile(coef_arr[:,i],25)>0 or
                               np.percentile(coef_arr[:,i],75)<0)
                     else 'No' for i in range(len(BEST_TRI_COLS))]
})
coef_df.to_csv('nested_WHO_v3_Table5_Coefficients.csv', index=False)
print('LR Coefficient Analysis:')
print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2E75B6' if m>0 else '#C0392B' for m in coef_df['Median']]
y_pos  = range(len(BEST_TRI_LABELS))
ax.barh(y_pos, coef_df['Median'], color=colors, alpha=0.8, height=0.5)
ax.errorbar(coef_df['Median'], y_pos,
            xerr=[coef_df['Median']-coef_df['Q25'], coef_df['Q75']-coef_df['Median']],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(y_pos); ax.set_yticklabels(BEST_TRI_LABELS, fontsize=11)
ax.set_xlabel('LR Coefficient (median ± IQR)', fontsize=11)
ax.set_title(f'Figure 2. LR Coefficients — {BEST_TRI_NAME}', fontsize=10)
plt.tight_layout()
plt.savefig('nested_WHO_v3_Figure_Coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nested_WHO_v3_Figure_Coefficients.png')

## Table 6: Spearman Correlation

In [ ]:
corr_data = df[BEST_TRI_COLS].dropna()
corr_data.columns = BEST_TRI_LABELS
corr_matrix = corr_data.corr(method='spearman')
print('Spearman Correlation Matrix:')
print(corr_matrix.round(3).to_string())
print(f'Max |r|: {corr_matrix.abs().values[np.triu_indices(3,k=1)].max():.3f}')
corr_matrix.round(3).to_csv('nested_WHO_v3_Table6_Spearman.csv')

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            square=True, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'shrink':0.8}, annot_kws={'size':12})
ax.set_title(f'Figure 3. Spearman Correlation Matrix\n{BEST_TRI_NAME}', fontsize=10)
plt.tight_layout()
plt.savefig('nested_WHO_v3_Figure_Spearman.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nested_WHO_v3_Figure_Spearman.png')

## Figure 1: ROC Curves — Univariate → Bivariate → Trivariate

In [ ]:
best_uni_col = REVERSE[t1.iloc[0]['Biomarker']]
best_bi_cols = [REVERSE[s.strip()] for s in t2.iloc[0]['Pair'].split('+')]

models_roc = [
    (f'Univariate ({t1.iloc[0]["Biomarker"]})',  [best_uni_col]),
    (f'Bivariate ({t2.iloc[0]["Pair"]})',         best_bi_cols),
    (f'Trivariate ({BEST_TRI_NAME})',             BEST_TRI_COLS),
]
colors  = ['#5B9BD5', '#2E75B6', '#1F4E79']
all_fpr = np.linspace(0, 1, 100)

fig, ax = plt.subplots(figsize=(6, 6))
for (label, cols), color in zip(models_roc, colors):
    X_m, y_m = get_subset(df, cols)
    tpr_list, auc_list = [], []
    for i in range(N_SPLITS):
        Xtr, Xte, ytr, yte = train_test_split(
            X_m, y_m, test_size=TEST_SIZE, random_state=i, stratify=y_m)
        clf = LogisticRegression(max_iter=1000, random_state=i).fit(Xtr, ytr)
        ypr = clf.predict_proba(Xte)[:,1]
        fpr, tpr, _ = roc_curve(yte, ypr)
        tpr_list.append(np.interp(all_fpr, fpr, tpr))
        auc_list.append(roc_auc_score(yte, ypr))
    mean_tpr = np.mean(tpr_list, axis=0)
    med_auc  = round(np.median(auc_list), 3)
    ci_lo    = round(np.percentile(auc_list, 2.5), 3)
    ci_hi    = round(np.percentile(auc_list, 97.5), 3)
    ax.plot(all_fpr, mean_tpr, color=color, lw=2,
            label=f'{label}\n(AUC = {med_auc}, 95% CI [{ci_lo}–{ci_hi}])')

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Figure 1. Mean ROC Curves\n(WHO risk group, 100 splits)', fontsize=11)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig('nested_WHO_v3_Figure_ROC.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nested_WHO_v3_Figure_ROC.png')

## Sensitivity Analysis: Cross-endpoint WITHOUT SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE as _SMOTE

MASAOKA_BEST_COLS = [
    'EphA6 EPITHELIAL H-score Membranous',
    'YAP EPITHELIAL H-score Nuclear',
    'HDAC4_EPITHELIAL_H_SCORE Cytoplasmic',
]
print('Cross-endpoint: Masaoka best triad applied to WHO outcome')
print('='*60)

for use_smote, label in [(True,'WITH SMOTE'),(False,'WITHOUT SMOTE')]:
    sub = df[MASAOKA_BEST_COLS + [TARGET_COL]].dropna()
    Xm  = sub[MASAOKA_BEST_COLS].values.astype(float)
    ym  = sub[TARGET_COL].values
    aucs, recs = [], []
    for i in range(N_SPLITS):
        Xtr, Xte, ytr, yte = train_test_split(
            Xm, ym, test_size=TEST_SIZE, random_state=i, stratify=ym)
        if use_smote:
            mc = int((ytr==1).sum())
            if mc < 2: continue
            sm = _SMOTE(random_state=i, k_neighbors=min(5,mc-1))
            try: Xtr, ytr = sm.fit_resample(Xtr, ytr)
            except: continue
        clf = LogisticRegression(max_iter=1000, random_state=i).fit(Xtr, ytr)
        try:
            aucs.append(roc_auc_score(yte, clf.predict_proba(Xte)[:,1]))
            recs.append(recall_score(yte, clf.predict(Xte), zero_division=0))
        except: pass
    print(f'\n  {label} (n={len(sub)}, {len(aucs)} valid splits):')
    print(f'    AUC    = {np.median(aucs):.3f}  95%CI [{np.percentile(aucs,2.5):.3f}–{np.percentile(aucs,97.5):.3f}]')
    print(f'    Recall = {np.median(recs):.3f}')
print('\n→ If results are similar, SMOTE choice does not drive the cross-endpoint finding.')

## Supplementary Figures S1, S2, S3

In [ ]:
# ── Supplementary S1: Binary WHO groups ──────────────────────────────────────
plot_df = df[BEST_TRI_COLS + [TARGET_COL]].dropna().copy()
plot_df['WHO Risk Group'] = plot_df[TARGET_COL].map({0:'Low-risk',1:'High-risk'})
palette = {'Low-risk':'#B5D4F4','High-risk':'#2E75B6'}
order   = ['Low-risk','High-risk']
n_low   = (plot_df['WHO Risk Group']=='Low-risk').sum()
n_high  = (plot_df['WHO Risk Group']=='High-risk').sum()

fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))
for ax, col, label in zip(axes, BEST_TRI_COLS, BEST_TRI_LABELS):
    data = [plot_df[plot_df['WHO Risk Group']==g][col].astype(float).values for g in order]
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color='black',linewidth=1.8),
                    flierprops=dict(marker='o',markerfacecolor='gray',markersize=4,
                                   alpha=0.5,linestyle='none'), widths=0.5)
    for patch, grp in zip(bp['boxes'], order):
        patch.set_facecolor(palette[grp]); patch.set_alpha(0.85)
    ax.set_title(label, fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('H-score (0–300)', fontsize=10)
    ax.set_xticks([1,2])
    ax.set_xticklabels([f'Low-risk\n(A/AB/B1)\nn={n_low}',
                        f'High-risk\n(B2/B3/Ca)\nn={n_high}'], fontsize=10)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig.suptitle('Supplementary Figure S1. H-score distributions — optimal trivariate markers\n'
             'stratified by WHO histological risk group',
             fontsize=11, y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('nested_WHO_v3_Supplementary_S1_Boxplots_Binary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved S1')

In [ ]:
# ── Supplementary S2: WHO subtypes ───────────────────────────────────────────
who_order = ['A','AB','B1','B2','B3','Carcinoma']
palette6  = {'A':'#A8D5F5','AB':'#7BBFE8','B1':'#4FA8DA',
             'B2':'#F4A460','B3':'#E07B2A','Carcinoma':'#C0392B'}
df6 = df[df[SUBTYPE_COL].isin(who_order)].copy()
n6  = {s:(df6[SUBTYPE_COL]==s).sum() for s in who_order}

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, col, label in zip(axes, BEST_TRI_COLS, BEST_TRI_LABELS):
    df_col = df6[[SUBTYPE_COL, col]].dropna()
    data   = [df_col[df_col[SUBTYPE_COL]==s][col].astype(float).values for s in who_order]
    bp = ax.boxplot(data, patch_artist=True,
                    medianprops=dict(color='black',linewidth=1.8),
                    flierprops=dict(marker='o',markerfacecolor='gray',markersize=4,
                                   alpha=0.5,linestyle='none'), widths=0.55)
    for patch, s in zip(bp['boxes'], who_order):
        patch.set_facecolor(palette6[s]); patch.set_alpha(0.85)
    ax.set_title(label, fontsize=13, fontweight='bold', pad=10)
    ax.set_ylabel('H-score (0–300)', fontsize=10)
    ax.set_xticks(range(1,7))
    ax.set_xticklabels([f'{s}\n(n={n6[s]})' for s in who_order], fontsize=9)
    ax.set_ylim(-10, 315)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.axvspan(0.5,3.5,alpha=0.05,color='#2E75B6',zorder=0)
    ax.axvspan(3.5,6.5,alpha=0.05,color='#C0392B',zorder=0)
    ax.axvline(3.5,color='gray',linewidth=0.8,linestyle='--',alpha=0.6)
    ax.text(2.0,308,'Low-risk',ha='center',fontsize=8,color='#2E75B6',style='italic')
    ax.text(5.0,308,'High-risk',ha='center',fontsize=8,color='#C0392B',style='italic')

fig.suptitle('Supplementary Figure S2. H-score distributions — optimal trivariate markers\n'
             'stratified by WHO histological subtype',
             fontsize=11, y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('nested_WHO_v3_Supplementary_S2_Boxplots_Subtypes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved S2')

In [ ]:
# ── Supplementary S3: Decision Tree ──────────────────────────────────────────
X_tree, y_tree = get_subset(df, BEST_TRI_COLS)
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tree, y_tree)
print(f'Decision Tree — training accuracy (n={len(X_tree)}): {round(accuracy_score(y_tree, dt.predict(X_tree)),3)}')
print('Note: training accuracy is optimistic — exploratory visualisation only')
print()
print('Decision Tree rules:')
print(export_text(dt, feature_names=BEST_TRI_LABELS))

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(dt, feature_names=BEST_TRI_LABELS, class_names=['Low-risk','High-risk'],
          filled=True, rounded=True, fontsize=11, ax=ax, impurity=False, proportion=False)
ax.set_title(f'Supplementary Figure S3. Decision Tree (depth=3) — {BEST_TRI_NAME}\n'
             f'WHO risk group prediction (n={len(X_tree)}). '
             f'Exploratory visualisation only; primary model is Logistic Regression.',
             fontsize=11, pad=12)
plt.tight_layout()
plt.savefig('nested_WHO_v3_Supplementary_S3_DecisionTree.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved S3')

## Summary

In [ ]:
all_deltas = [t2.iloc[i]['ΔAUC'] for i in range(len(t2))] +               [t3.iloc[i]['ΔAUC'] for i in range(len(t3))]
lr_wins  = sum(1 for d in all_deltas if d > 0)
xgb_wins = sum(1 for d in all_deltas if d < 0)

print('=' * 65)
print('WHO NESTED v3 ANALYSIS — COMPLETE')
print('=' * 65)
print(f'\nBest univariate:  {t1.iloc[0]["Biomarker"]}  LR={t1.iloc[0]["LR_AUC"]}  95%CI={t1.iloc[0]["LR_AUC_95CI"]}')
print(f'Best bivariate:   {t2.iloc[0]["Pair"]}  LR={t2.iloc[0]["LR_AUC"]}  95%CI={t2.iloc[0]["LR_AUC_95CI"]}')
print(f'Best trivariate:  {BEST_TRI_NAME}  LR={BEST_TRI_AUC}')
best_tri_row = t3[t3['LR_AUC']==BEST_TRI_AUC].iloc[0]
print(f'                  95%CI={best_tri_row["LR_AUC_95CI"]}')
print(f'Best tetrad:      {t4.iloc[0]["Tetrad"]}  LR={t4.iloc[0]["AUC"]}')
print()
print(f'LR vs XGBoost: LR wins {lr_wins}/{len(all_deltas)}, XGB wins {xgb_wins}/{len(all_deltas)}')
print()
print('Nested top-5 frequency (confirms original selection):')
for m in sorted(top5_freq, key=top5_freq.get, reverse=True)[:5]:
    print(f'  {SHORT_NAMES.get(m,m):<30} {top5_freq[m]}/100')
print()
print('Files saved:')
for f in [
    'nested_WHO_v3_Table1_Univariate.csv',
    'nested_WHO_v3_Table2_Bivariate.csv',
    'nested_WHO_v3_Table3_Trivariate.csv',
    'nested_WHO_v3_Table4_Tetrads.csv',
    'nested_WHO_v3_Table5_Coefficients.csv',
    'nested_WHO_v3_Table6_Spearman.csv',
    'nested_WHO_v3_Figure_ROC.png',
    'nested_WHO_v3_Figure_Coefficients.png',
    'nested_WHO_v3_Figure_Spearman.png',
    'nested_WHO_v3_Supplementary_S1_Boxplots_Binary.png',
    'nested_WHO_v3_Supplementary_S2_Boxplots_Subtypes.png',
    'nested_WHO_v3_Supplementary_S3_DecisionTree.png',
]:
    print(f'  {f}')